# 文学研究のためのメタデータ実習

2026年10月16日 技術講習会

Open Libraryの書誌情報、Wikidataの照合結果、OpenAlexの暫定候補数を検索します。

1. 「ランタイム」からすべてのセルを実行します。
2. 作品名・著者名を入力して検索します。
3. Edition数とOpenAlexの状態フラグを確認します。

**注意：** OpenAlexの候補数は検証済みの研究論文数ではありません。
未解決グループの参考値は、通常の統合集計と直接比較できません。

この候補版のデータURLは、CSVをGitHubへ公開するまで有効ではありません。


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO = "haruka0221/literary-metadata-workshop-2026"
REF = "main"
URL = f"https://raw.githubusercontent.com/{REPO}/{REF}/data/workshop_search_v1.csv"

try:
    df = pd.read_csv(URL, dtype=str, keep_default_na=False, encoding="utf-8-sig")
except Exception as exc:
    raise RuntimeError("講習会用CSVがまだ公開されていないか、取得できません: " + URL) from exc

FLAG_LABELS = {
    "PROVISIONAL": "暫定（作品ID接続済み）",
    "PROVISIONAL_UNRESOLVED": "暫定・作品同定未確定",
    "NOT_AVAILABLE": "候補数を表示できない",
    "IDENTITY_HOLD": "作品同定保留",
}

required = {
    "ol_work_id", "ol_title", "ol_author_name",
    "ol_first_publish_year", "ol_edition_count",
    "openalex_display_candidate_count", "openalex_quality_flag",
    "openalex_count_unit_id", "wikidata_decision",
}
assert required <= set(df.columns), "CSVの列が想定と異なります"
assert len(df) == 34789, "CSVの行数が想定と異なります"

print(f"読み込み成功：{len(df):,} OLレコード")
display(df.head(3))


## 1. 作品のタイトル・著者から検索する
部分一致で検索します。空欄を指定すると、その条件では絞り込みません。

In [ ]:
TITLE = "The Great Gatsby" # @param {type:"string"}
AUTHOR = "" # @param {type:"string"}

q = df.copy()

if TITLE.strip():
    q = q[q["ol_title"].str.contains(TITLE.strip(), case=False, regex=False, na=False)]
if AUTHOR.strip():
    q = q[q["ol_author_name"].str.contains(AUTHOR.strip(), case=False, regex=False, na=False)]

q = q.copy()
q["_edition_sort"] = pd.to_numeric(q["ol_edition_count"], errors="coerce").fillna(-1)
q = q.sort_values("_edition_sort", ascending=False)
q["openalex_status_japanese"] = q["openalex_quality_flag"].map(FLAG_LABELS).fillna("要確認")

print(f"検索結果：{len(q):,} OLレコード")
display(q[[
    "ol_title", "ol_author_name", "ol_work_id",
    "ol_first_publish_year", "ol_edition_count",
    "wikidata_decision", "openalex_display_candidate_count",
    "openalex_status_japanese",
]].head(30))

print("同じ作品と思われる複数のOLレコードが表示される場合があります。")
print("OpenAlex候補数をこれらの行で合算しないでください。")


## 2. 年代で作品の集合を見る
指定する年はOpen Libraryのソース上の最初の出版年で、厳密な文学史上の初出年とは限りません。このCSVは歴史的OL対象34,789レコードを収録しますが、すべての英語小説を網羅するものではありません。

In [ ]:
START_YEAR = 1880 # @param {type:"integer"}
END_YEAR = 1900 # @param {type:"integer"}
years = pd.to_numeric(df["ol_first_publish_year"], errors="coerce")
period = df[years.between(START_YEAR, END_YEAR)].copy()
print(f"この読み込みデータ内で{START_YEAR}–{END_YEAR}年に該当：{len(period):,} OLレコード")
display(period[["ol_title","ol_author_name","ol_first_publish_year"]].head(25))

## 3. Edition数を比較する
この列が入力CSVで欠損の場合は、数値が記録されていないことを示します。Editionは実際の刊行回数・刷数・売上・読者数ではありません。

In [ ]:
PLOT_TOP = 15 # @param {type:"integer", min:5, max:40}
x = period.copy()
x["edition_numeric"] = pd.to_numeric(x["ol_edition_count"], errors="coerce")
x = x.dropna(subset=["edition_numeric"]).nlargest(PLOT_TOP,"edition_numeric")
if x.empty: print("このデータにはEdition数が記録されていません。")
else:
    display(x[["ol_title","ol_author_name","ol_edition_count"]])
    ax=x.sort_values("edition_numeric").plot.barh(x="ol_title",y="edition_numeric",legend=False,figsize=(9,max(4,len(x)*.38)))
    ax.set(xlabel="Registered OL Edition records",ylabel="OL title"); plt.tight_layout(); plt.show()

## 4. Wikidataとの照合状況
MATCHは採用された照合、AMBIGUOUSは曖昧、NO_MATCHは現行の探索・判定手続きでは照合できなかったことを意味します。

In [ ]:
d = df["wikidata_decision"].replace("", "NOT_RECORDED").value_counts(dropna=False)
display(d.to_frame("OL source records"))
d.plot.bar(figsize=(7,3),rot=0,title="Wikidata resolution status");plt.tight_layout();plt.show()

## 5. OpenAlexの暫定候補数と注意フラグ

検索済みの作品について、OpenAlexの候補数とその状態を確認します。

- `PROVISIONAL`：Project Workに接続済みだが、候補数は暫定的。
- `PROVISIONAL_UNRESOLVED`：作品同定未確定。別の検索方法による参考値。
- `NOT_AVAILABLE`：表示できる候補数がない。0件という意味ではない。
- `IDENTITY_HOLD`：作品の同定に問題があり保留。

異なる方法で得た候補数を、そのまま比較・合算しないでください。


In [ ]:
print("=== 全体の状態フラグ ===")
display(
    df["openalex_quality_flag"]
    .value_counts()
    .rename_axis("状態")
    .reset_index(name="OLレコード数")
)

print("\n=== 検索した作品のOpenAlex候補 ===")
display(q[[
    "ol_title", "ol_work_id",
    "openalex_display_candidate_count",
    "openalex_quality_flag", "openalex_count_source",
    "openalex_count_unit_id", "openalex_qa_note",
]].head(30))

print("同じ計測単位IDに同じ候補数が繰り返される場合があります。")
print("同じ数値を複数作品分として加算しないでください。")


## 6. 観察と解釈

- Edition数の多さは、読者が多かったことを意味するでしょうか。出版年・出版社・収録時点の偏りを考えてみましょう。
- WikidataでNO_MATCHだった作品は「存在しない作品」でしょうか。
- OpenAlex候補が多い作品は、学術的評価が高いと断定できるでしょうか。同名作品・一般語タイトル・タイトルと著者の誤結合が起こりえます。
- Open Library Workと研究上のProject Workは同一の概念でしょうか。今回の17件の作品同定保留事例を考えてみましょう。

### 発展：一作品のEditionを深く見る
この配布CSVは主にWorkごとのメタデータです。Editionの出版社・刊行年・言語を年表で扱うためには、事前に監査した別のEditionテーブルが必要です。未収集の詳細をこのCSVから補完したように表示しません。